# 분류를 위해 미세 튜닝하기
지금까지 LLM 구조를 구현하고 사전 훈련했다. 오픈 AI 같이 사전 훈련된 가중치를 직접 만든 모델에 로드하는 방법을 배웠다. 이제 텍스트 분류 같은 특정 타깃 작업에 LLM을 미세튜닝하면 이런 작업의 가치가 드러난다. 이 장에서 다룰 구체적인 예제는 텍스트 메시지를 '스팸' 또는 '스팸 아님'으로 분류하는 것이다.

## 6.1 여러 가지 미세 튜닝 방법
언어 모델을 미세 튜닝하는 가장 일반적인 방법은 **지시 미세 튜닝(instruction fine-tuning)** 과 **분류 미세 튜닝(classification fine-tuning)** 이다. 지시 미세 튜닝은 구체적인 지시 데이터를 사용해 일련의 작업에서 언어 모델을 훈련한다.

분류 미세 튜닝에서는 모델이 '스팸'과 '스팸 아님' 같은 일련의 클래스 레이블을 인식하도록 훈련한다. 분류 작업의 예는 LLM과 이메일 필터링에 국한되지 않는다. 이미지에 있는 다양한 식물의 종을 식별하거나, 뉴스 기사를 스포츠, 정치, 기술 같은 토픽으로 분류하거나, 의료 이미지를 통해 양성 종양과 악성 종양을 구분한다.

핵심은 분류 미세 튜닝 모델이 훈련 과정에서 만난 클래스만 예측한다는 것이다. 에를 들어 모델은 입력 텍스트가 '스팸' 또는 '스팸 아님'인지 결정할 수 있지만 그외 다른 것은 말할 수 없다.

분류를 위해 미세 튜닝된 모델과 달리 지시 미세 튜닝 모델은 일반적으로 광범위한 작업을 수행할 수 있다. 분류 미세 튜닝된 모델은 고도로 전문화되었다고 볼 수 있으며, 일반적으로 다양한 작업을 처리하는 일반화된 모델보다 전문화된 모델을 개발하는 것이 더 쉽다.

## 6.2 데이터셋 준비
앞서 구현하고 사전 훈련한 GPT 모델을 수정해서 분류 미세튜닝을 수행할 수 있다. 분류 미세 튜닝에 대한 유용한 예제로 '스팸'과 '스팸 아님'으로 구성된 텍스트 메시지 데이터셋을 사용한다.

먼저 데이터셋을 다운로드한다.

In [1]:
import requests
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path}가 이미 있어 다운로드 및 압축 해제를 건너뜁니다.")
        return

    # 파일을 다운로드 합니다.
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    with open(zip_path, "wb") as out_file:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                out_file.write(chunk)

    # 파일 압축을 풉니다.
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # .tsv 파일 확장자를 추가합니다.
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"파일이 다운로드되어 {data_file_path}에 저장되었습니다.")

try:
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (requests.exceptions.RequestException, TimeoutError) as e:
    print(f"기본 URL 실패: {e}. 백업 URL을 시도합니다...")
    url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

파일이 다운로드되어 sms_spam_collection/SMSSpamCollection.tsv에 저장되었습니다.


In [3]:
import pandas as pd
df = pd.read_csv(
    data_file_path, sep="\t", header=None, names=["Label", "Text"]
)
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


클래스 레이블 분포를 조사해보자.

In [4]:
print(df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


데이터에 "ham"(즉, 스팸 아님)이 "spam"보다 훨씬 많다는 것을 알 수 있다.

LLM 미세 튜닝을 빠르게 수행하기 위해 작은 데이터셋이 좋으므로 각 클래스에 대해 747개의 샘플만 포함하도록 데이터셋을 줄인다.

다음 코드를 사용해 데이터셋을 언더샘플링(undersapmling)하여 균형잡힌 데이터셋을 만든다.

In [6]:
def create_balanced_dataset(df):
  num_spam = df[df["Label"] == "spam"].shape[0] # "spam" 샘플 개수를 카운트한다
  ham_subset = df[df["Label"] == "ham"].sample(
      num_spam, random_state=123
  ) # "spam" 샘플 개수만큼 "ham" 샘플을 랜덤하게 선택한다
  balanced_df = pd.concat([
      ham_subset, df[df["Label"] == "spam"]
  ]) # 선택된 샘플과 "spam" 샘플을 합친다
  return balanced_df

balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())


Label
ham     747
spam    747
Name: count, dtype: int64


스팸 메시지와 스팸이 아닌 메시지의 개수가 동일해진다.

그런 다음 문자열로 된 클래스 레이블 "ham"과 "spam"을 정수 클래스 레이블 0과 1로 각각 변경한다.

In [7]:
balanced_df["Label"] = balanced_df["Label"].map({"ham":0, "spam":1})

50,000개 단어 이상으로 구성된 GPT 어휘사전을 사용하지 않고 딱 2개의 토큰 ID와 0과 1을 사용한다.

그런 다음 데이터셋을 세 부분으로 분할하는 random_split 함수를 구현한다. 70%는 훈련용, 10%는 검증용, 20%는 테스트용으로 분할한다.

In [8]:
def random_split(df, train_frac, validation_frac):

  df = df.sample(
      frac=1, random_state=123
  ).reset_index(drop=True) # 전체 프레임을 섞는다
  train_end = int(len(df) * train_frac) # 분할할 인덱스를 계산한다
  validation_end = train_end + int(len(df) * validation_frac)

  # 데이터프레임을 분할한다
  train_df = df[:train_end]
  validation_df = df[train_end:validation_end]
  test_df = df[validation_end:]

  return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(
    balanced_df, 0.7, 0.1) # 테스트 크기는 나머지에 해당하는 0.2이다

이 데이터셋을 나중에 재사용할 수 있도록 CSV 파일로 저장해보자.

In [9]:
train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

지금까지 데이터셋을 다운로드하고, 클래스 균형을 맞추고, 훈련/검증/평가 세트로 나누었다. 이제 모델 훈련에 사용할 파이토치 데이터 로더를 준비해보자.

## 6.3 데이터 로더 만들기
텍스트 데이터로 작업할 때 구현했던 것과 개념적으로 비슷한 파이토치 데이터 로더를 만든다. 이전에는 슬라이딩 윈도 기법을 사용해 균일한 크기의 텍스트 청크를 생성했다. 효율적인 모델 훈련을 위해 이를 배치로 묶었다. 각 청크는 개별 훈련 샘플이 된다. 하지만 여기서는 사용하는 스팸 데이터셋은 길이가 다양하다. 텍스트 청크로 했던 것처럼 이런 메시지를 배치로 묶으려면 두 가지 방식이 있다.
- 데이터셋이나 배치에 있는 가장 짧은 길이의 메시지에 맞춰 모든 메시지를 잘라낸다.
- 데이터셋이나 배치에 있는 가장 긴 길이의 메시지에 맞춰 모든 메시지에 패딩을 추가한다.

첫 번째 방식은 계산 비용이 저렴하다. 하지만 가장 짧은 메시지는 평균이나 가장 긴 메시지에 비해 길이가 훨씬 짧기 때문에 정보 손실이 많을 수 있어 모델의 성능이 낮아진다. 따라서 메시지 내용을 모두 보존하는 두 번째 방식을 선택한다.

데이터셋에서 가장 긴 메시지 길이로 모든 메시지를 맞추어 배치를 만들기 위해 짧은 길이의 메시지에 패딩 토큰을 추가한다. 이를 위해 "<|endoftext|>"를 패딩 토큰으로 사용한다.

하지만 각 원본 텍스트 메시지에 문자열 "<|endoftext|>"를 추가하는 대신 인코딩된 텍스트 메시지에 "<|endoftext|>"에 해당하는 토큰 ID를 추가할 수 있다. 50256은 패딩 토큰 "<|endoftext|>"에 해당하는 토큰ID이다.

우선 tik token 패키지의 GPT-2 토크나이저를 사용해 "<|endoftext|>"의 토큰ID가 50256이 맞는지 확인해보자.

In [10]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


먼저 데이터 로더를 만들기 전에 데이터를 로드하고 처리하는 방법을 저장하기 위해 파이토치의 Dataset을 구현해야 한다. 이를 위해 SpamDataset 클래스를 정의한다. SpamDataset 클래스는 몇 가지 주요 작업을 처리한다. 텍스트 메시지를 토큰 시퀀스로 인코딩하고, 훈련 데이터셋에서 가장 긴 시퀀스를 식별하고, 다른 모든 시퀀스에 패딩 토큰을 추가하여 가장 긴 시퀀스 길이에 맞춘다.

In [11]:
import torch
from torch.utils.data import Dataset

class SpamDataset(Dataset):
  def __init__(self, csv_file, tokenizer, max_length=None,
               pad_token_id=50256):
    self.data = pd.read_csv(csv_file)

    self.encoded_texts = [ # 텍스트를 토큰화한다
        tokenizer.encode(text) for text in self.data["Text"]
    ]

    if max_length is None:
      self.max_length = self._longest_encoded_length()
    else:
      self.max_length = max_length # max_length보다 긴 시퀀스를 자른다

      self.encoded_texts = [
          encoded_text[:self.max_length]
          for encoded_text in self.encoded_texts
      ]

    self.encoded_texts = [
        encoded_text + [pad_token_id] *(self.max_length-len(encoded_text))
        for encoded_text in self.encoded_texts
    ] # 가장 긴 시퀀스에 맞춰 패딩을 추가한다.

  def __getitem__(self, index):
    encoded = self.encoded_texts[index]
    label = self.data.iloc[index]["Label"]
    return (
        torch.tensor(encoded, dtype=torch.long),
        torch.tensor(label, dtype=torch.long)
    )

  def __len__(self):
    return len(self.data)

  def _longest_encoded_length(self):
    max_length = 0
    for encoded_text in self.encoded_texts:
      encoded_length = len(encoded_text)
      if encoded_length > max_length:
        max_length = encoded_length
    return max_length

SampleDataset 클래스는 앞서 만든 CSV 파일에서 데이터를 로드하고, tiktoken의 GPT-2 토크나이저를 사용해 텍스트를 토큰화하고, 가장 긴 시퀀스나 사전에 정의된 최대 길이에 맞춰 동일한 길이가 되도록 시퀀스를 자르고 패딩을 추가한다. 이렇게 하면 입력 텐서가 동일한 길이가 된다. 이는 다음에 구현할 훈련 데이터 로더에서 배치를 만들기 위해 필수적인 작업이다.

In [12]:
train_dataset = SpamDataset(
    csv_file = "train.csv",
    max_length = None,
    tokenizer = tokenizer
)

가장 긴 시퀀스 길이는 이 데이터셋의 max_length 속성에 저장되어 있다.

In [13]:
print(train_dataset.max_length)

120


즉, 가장 긴 시퀀스에 120개보다 많은 토큰이 들어 있지 않다.

그런 다음 가장 긴 훈련 세트의 시퀀스 길이에 맞춰 검증 세트와 테스트 세트에 패딩을 추가한다. 중요한 것은 가장 긴 훈련 샘플의 길이보다 긴 모든 검증 세트 샘플과 테스트 세트 샘플은 SpamDataset에 있는 encoded_text[:self.max_length]에 의해 잘라야 한다는 점이다.

In [14]:
val_dataset = SpamDataset(
    csv_file="validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

test_dataset = SpamDataset(
    csv_file="test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

이 경우에는 타깃이 텍스트의 다음 토큰이 아니라 클래스 레이블이다. (0 또는 1)

다음 코드는 텍스트 메시지와 레이블을 로드하여 배치 크기 8인 훈련, 검증, 테스트 세트 데이터 로더를 만든다.

In [16]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8
torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False
)

데이터 로더가 기대한 크기의 배치를 반환하는지 확인하기 위해 훈련 데이터 로더를 반복하면서 마지막 배치의 텐서 차원을 출력해보자.

In [17]:
for input_batch, target_batch in train_loader:
  pass
print("입력 배치 차원:", input_batch.shape)
print("레이블 배치 차원:", target_batch.shape)

입력 배치 차원: torch.Size([8, 120])
레이블 배치 차원: torch.Size([8])


여기서 볼 수 있듯이 입력 배치는 8개의 샘플로 구성되며, 각 샘플은 120개 토큰을 가진다. 레이블 텐서는 8개의 훈련 샘플에 해당하는 클래스 레이블을 저장하고 있다.

마지막으로 데이터셋 크기를 확인하기 위해 각 데이터셋에 있는 전체 배치 개수를 출력해보자.

In [18]:
print(f"{len(train_loader)}개 훈련 배치")
print(f"{len(val_loader)}개 검증 배치")
print(f"{len(test_loader)}개 테스트 배치")

130개 훈련 배치
19개 검증 배치
38개 테스트 배치


이제 데이터가 준비되었으므로 미세 튜닝을 위한 모델을 준비할 차례이다.

## 6.4 사전 훈련된 가중치로 모델 초기화하기
스팸 메시지를 식별하도록 분류 미세 튜닝하기 위한 모델을 준비해야 한다.

먼저 사전 훈련된 모델을 초기화한다.

In [20]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every efforts moves"
BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True
}
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])